In [3]:
# CELL 1: Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


In [4]:
# CELL 2: Unzip — skips if already extracted
import os, zipfile

zip_path     = '/content/drive/MyDrive/AVEC2014.zip'
extract_path = '/content/avec2014'
marker       = os.path.join(extract_path, 'AVEC2014', 'labels.csv')

if os.path.exists(marker):
    print('Already extracted — skipping.')
else:
    print('Extracting...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)
    print('Done.')

Extracting...
Done.


In [5]:
# CELL 3: Imports and constants
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import subprocess
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print(subprocess.getoutput('nvidia-smi | grep "GPU 0"'))
print(f'GPU total: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# ImageNet normalisation — supervisor specified, standard for Sports-1M C3D
# Applied as: (frames / 255.0 - mean) / std
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

DATA_ROOT  = '/content/avec2014/AVEC2014'
FRAME_ROOT = '/content/avec2014_frames'
CLIP_LEN   = 16
STRIDE     = 8    # paper: 8-frame overlap between consecutive clips
BATCH_SIZE = 16   # A100: 32 | L4: 4 | H100: 64

Device: cuda

GPU total: 85.1 GB


In [ ]:
# CELL 4: Keep-alive — prevents Colab disconnecting during long training
import time, threading

def keep_alive():
    while True:
        time.sleep(60)
        print('.', end='', flush=True)

t = threading.Thread(target=keep_alive, daemon=True)
t.start()
print('Keep-alive started.')

Keep-alive started.


In [10]:
# CELL 5: Load labels and gender map
def load_labels():
    df = pd.read_csv(os.path.join(DATA_ROOT, 'labels.csv'))
    labels = {}
    for _, row in df.iterrows():
        key = str(row['filename']).strip().replace('\\', '/')
        key = os.path.splitext(key)[0]
        labels[key] = float(row['BDI-II'])
    print(f'Labels loaded: {len(labels)}')
    print('Sample keys:', list(labels.keys())[:3])
    return labels

def load_gender_map():
    path = os.path.join(DATA_ROOT, 'gender.csv')
    if not os.path.exists(path):
        print('No gender.csv — gender breakdown skipped.')
        return {}
    df = pd.read_csv(path)
    gmap = {}
    for _, row in df.iterrows():
        key = str(row['filename']).strip().replace('\\', '/')
        key = os.path.splitext(key)[0]
        gmap[key] = str(row['gender']).strip().upper()
    return gmap

labels     = load_labels()
gender_map = load_gender_map()

Labels loaded: 300
Sample keys: ['Training/Northwind/203_1_Northwind_video', 'Training/Freeform/203_1_Freeform_video', 'Training/Northwind/205_2_Northwind_video']


In [8]:
!pip install deepface -q

from deepface import DeepFace
import cv2, os, pandas as pd
from tqdm import tqdm

DATA_ROOT = '/content/avec2014/AVEC2014'

def annotate_video_gender_age(path, stem, sample_every=30):
    cap = cv2.VideoCapture(path)
    genders, ages = [], []
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % sample_every == 0:
            try:
                result = DeepFace.analyze(
                    frame,
                    actions=['age', 'gender'],
                    enforce_detection=False,
                    silent=True
                )
                genders.append(result[0]['dominant_gender'])
                ages.append(result[0]['age'])
            except Exception:
                pass
        idx += 1
    cap.release()
    if not genders:
        return None, None
    gender   = 'M' if genders.count('Man') > genders.count('Woman') else 'F'
    age      = float(sum(ages) / len(ages))
    return gender, age

all_items = train_items + val_items + test_items
rows = []
print(f'Annotating {len(all_items)} videos...')
for path, stem, label in tqdm(all_items):
    gender, age = annotate_video_gender_age(path, stem)
    rows.append({
        'filename':  stem,
        'gender':    gender if gender else '?',
        'age':       round(age, 1) if age else None,
        'age_group': 'Young' if age and age < 30 else 'Old'
    })

df = pd.DataFrame(rows)
out_path = os.path.join(DATA_ROOT, 'gender.csv')
df.to_csv(out_path, index=False)
# Also save to Drive so it persists across sessions
df.to_csv('/content/drive/MyDrive/avec2014_gender.csv', index=False)
print(f'Saved to DATA_ROOT and Drive.')
print(df['gender'].value_counts())

Annotating 300 videos...


  0%|          | 0/300 [00:00<?, ?it/s]

26-06-07 13:32:36 - 🔗 age_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/age_model_weights.h5 to /root/.deepface/weights/age_model_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/age_model_weights.h5
To: /root/.deepface/weights/age_model_weights.h5

  0%|          | 0.00/539M [00:00<?, ?B/s]
  2%|▏         | 11.0M/539M [00:00<00:06, 79.1MB/s]
  4%|▍         | 21.5M/539M [00:00<00:06, 81.0MB/s]
  6%|▌         | 32.0M/539M [00:00<00:06, 83.6MB/s]
  8%|▊         | 42.5M/539M [00:00<00:06, 80.5MB/s]
 10%|▉         | 53.0M/539M [00:00<00:06, 76.2MB/s]
 12%|█▏        | 63.4M/539M [00:00<00:06, 70.5MB/s]
 14%|█▎        | 73.9M/539M [00:00<00:06, 72.0MB/s]
 16%|█▌        | 84.4M/539M [00:01<00:06, 67.4MB/s]
 18%|█▊        | 94.9M/539M [00:01<00:06, 72.5MB/s]
 19%|█▉        | 103M/539M [00:01<00:07, 60.5MB/s] 
 20%|██        | 110M/539M [00:01<00:08, 48.4MB/s]
 21%|██▏       | 115M/539M [00:01<00:09, 46.1MB/s]
 22%|██▏       | 121M/539M [00:01<00:08, 46.5MB/s]
 23%|██▎       | 126M/539M [00:02<00:08, 48.8MB/s]
 25%|██▌       | 137M/539M [00:02<00:07, 54.9MB/s]
 26%|██▋       | 143M/5

26-06-07 13:32:51 - 🔗 gender_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/gender_model_weights.h5 to /root/.deepface/weights/gender_model_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/gender_model_weights.h5
To: /root/.deepface/weights/gender_model_weights.h5

  0%|          | 0.00/537M [00:00<?, ?B/s]
  2%|▏         | 8.91M/537M [00:00<00:07, 74.6MB/s]
  3%|▎         | 16.8M/537M [00:00<00:08, 60.9MB/s]
  4%|▍         | 23.1M/537M [00:00<00:08, 59.1MB/s]
  6%|▌         | 32.0M/537M [00:00<00:08, 60.4MB/s]
  8%|▊         | 42.5M/537M [00:00<00:07, 68.9MB/s]
 10%|▉         | 53.0M/537M [00:00<00:06, 73.1MB/s]
 12%|█▏        | 63.4M/537M [00:00<00:07, 65.2MB/s]
 14%|█▍        | 73.9M/537M [00:01<00:06, 69.1MB/s]
 15%|█▌        | 81.3M/537M [00:01<00:06, 67.6MB/s]
 16%|█▋        | 88.6M/537M [00:01<00:06, 65.3MB/s]
 18%|█▊        | 95.4M/537M [00:01<00:07, 63.1MB/s]
 19%|█▉        | 104M/537M [00:01<00:06, 66.1MB/s] 
 21%|██        | 111M/537M [00:01<00:06, 62.0MB/s]
 22%|██▏       | 117M/537M [00:01<00:08, 52.3MB/s]
 24%|██▎       | 126M/537M [00:01<00:06, 59.7MB/s]
 25%|██▌       

Saved to DATA_ROOT and Drive.
gender
M    195
F    105
Name: count, dtype: int64


In [7]:
# CELL 6: Collect videos
# Paper split for AVEC 2014:
#   Train : Training/ + Development/  (200 videos)
#   Val   : Testing/Northwind/         (50 videos)
#   Test  : Testing/Freeform/          (50 videos)

def collect_videos(folders, labels):
    if isinstance(folders, str):
        folders = [folders]
    items = []
    for folder in folders:
        if not os.path.exists(folder):
            print(f'WARNING: not found: {folder}')
            continue
        for root, _, files in os.walk(folder):
            for f in sorted(files):
                if not f.lower().endswith('.mp4'):
                    continue
                path = os.path.join(root, f)
                rel  = os.path.relpath(path, DATA_ROOT).replace('\\', '/')
                stem = os.path.splitext(rel)[0]
                if stem in labels:
                    items.append((path, stem, labels[stem]))
                else:
                    print(f'  No label for: {stem}')
    return items

TRAIN_DIRS = [
    os.path.join(DATA_ROOT, 'Training'),
    os.path.join(DATA_ROOT, 'Development'),
]
VAL_DIR  = os.path.join(DATA_ROOT, 'Testing', 'Northwind')
TEST_DIR = os.path.join(DATA_ROOT, 'Testing', 'Freeform')

train_items = collect_videos(TRAIN_DIRS, labels)
val_items   = collect_videos(VAL_DIR,    labels)
test_items  = collect_videos(TEST_DIR,   labels)

print(f'Train videos : {len(train_items)}  (expected 200)')
print(f'Val   videos : {len(val_items)}    (expected 50)')
print(f'Test  videos : {len(test_items)}   (expected 50)')

# Label normalisation — computed from training set only
train_labels_arr = np.array([l for _, _, l in train_items])
LABEL_MEAN = float(train_labels_arr.mean())
LABEL_STD  = float(train_labels_arr.std())
print(f'Label normalisation — mean: {LABEL_MEAN:.2f}  std: {LABEL_STD:.2f}')

Train videos : 200  (expected 200)
Val   videos : 50    (expected 50)
Test  videos : 50   (expected 50)
Label normalisation — mean: 15.34  std: 12.07


In [18]:
# CELL 7: Pre-extract frames to local SSD (skips if already done)
os.makedirs(FRAME_ROOT, exist_ok=True)

def extract_video(path, stem):
    out_dir = os.path.join(FRAME_ROOT, stem.replace('/', '_'))
    if os.path.exists(out_dir) and len(os.listdir(out_dir)) >= CLIP_LEN:
        return
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(path)
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (112, 112))
        cv2.imwrite(
            os.path.join(out_dir, f'{idx:05d}.jpg'),
            frame,
            [cv2.IMWRITE_JPEG_QUALITY, 95]
        )
        idx += 1
    cap.release()

all_items = train_items + val_items + test_items
print(f'Extracting frames for {len(all_items)} videos...')
for path, stem, label in tqdm(all_items):
    extract_video(path, stem)
print('Extraction complete.')

Extracting frames for 300 videos...


100%|██████████| 300/300 [05:03<00:00,  1.01s/it]

Extraction complete.


In [14]:
# CELL 8: Dataset classes
# Normalisation: (frames / 255.0 - mean) / std — supervisor specified
# Training labels normalised to zero mean/unit std to prevent mean collapse
# Predictions denormalised back to BDI-II scale at eval time

def load_frames_from_ssd(stem, start, n=CLIP_LEN):
    out_dir = os.path.join(FRAME_ROOT, stem.replace('/', '_'))
    frames  = []
    for i in range(start, start + n):
        fpath = os.path.join(out_dir, f'{i:05d}.jpg')
        if not os.path.exists(fpath):
            break
        frame = cv2.imread(fpath)
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    return frames

def count_frames(stem):
    out_dir = os.path.join(FRAME_ROOT, stem.replace('/', '_'))
    if not os.path.exists(out_dir):
        return 0
    return len([f for f in os.listdir(out_dir) if f.endswith('.jpg')])

def frames_to_tensor(frames):
    # Normalisation: (frames / 255.0 - mean) / std
    arr = np.stack(frames, axis=0).astype(np.float32) / 255.0
    arr = (arr - MEAN) / STD
    arr = arr.transpose(3, 0, 1, 2)  # (T,H,W,3) -> (3,T,H,W)
    return torch.from_numpy(arr)

def get_clip_starts(T, stride=STRIDE):
    return list(range(0, T - CLIP_LEN + 1, stride))


class AVEC2014Train(Dataset):
    def __init__(self, items):
        self.samples = []  # (stem, start, label)
        for path, stem, label in items:
            T = count_frames(stem)
            if T < CLIP_LEN:
                continue
            for s in get_clip_starts(T):
                self.samples.append((stem, s, label))
        print(f'Training clips: {len(self.samples)}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        stem, start, label = self.samples[idx]
        frames = load_frames_from_ssd(stem, start, CLIP_LEN)
        if len(frames) < CLIP_LEN:
            while len(frames) < CLIP_LEN:
                frames.append(frames[-1])
        norm_label = (label - LABEL_MEAN) / LABEL_STD
        return frames_to_tensor(frames), torch.tensor(norm_label, dtype=torch.float32)


class AVEC2014Eval(Dataset):
    def __init__(self, items):
        self.samples = []  # (stem, raw_label)
        for path, stem, label in items:
            T = count_frames(stem)
            if T >= CLIP_LEN:
                self.samples.append((stem, label))
        print(f'Eval videos: {len(self.samples)}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        stem, label = self.samples[idx]
        T     = count_frames(stem)
        clips = []
        for s in get_clip_starts(T, stride=8):
            frames = load_frames_from_ssd(stem, s, CLIP_LEN)
            if len(frames) < CLIP_LEN:
                continue
            clips.append(frames_to_tensor(frames))
        if not clips:
            clips.append(torch.zeros(3, CLIP_LEN, 112, 112))
        return torch.stack(clips), torch.tensor(label, dtype=torch.float32), stem

In [15]:
# CELL 9: C3D model
# Architecture matches pretrained Sports-1M checkpoint exactly:
#   pool5 padding=(0,1,1) produces spatial output 1x4x4 for 112x112 input
#   fc6 input = 512*1*4*4 = 8192 — matches checkpoint classifier.0 shape
#   fc7 penultimate = 64 per paper (Section IV-B-1)
#   fc8 output = 1 (regression)

class C3D(nn.Module):
    def __init__(self, dropout=0.5):
        super().__init__()
        self.conv1  = nn.Conv3d(3,   64,  kernel_size=(3,3,3), padding=(1,1,1))
        self.pool1  = nn.MaxPool3d(kernel_size=(1,2,2), stride=(1,2,2))
        self.conv2  = nn.Conv3d(64,  128, kernel_size=(3,3,3), padding=(1,1,1))
        self.pool2  = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2))
        self.conv3a = nn.Conv3d(128, 256, kernel_size=(3,3,3), padding=(1,1,1))
        self.conv3b = nn.Conv3d(256, 256, kernel_size=(3,3,3), padding=(1,1,1))
        self.pool3  = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2))
        self.conv4a = nn.Conv3d(256, 512, kernel_size=(3,3,3), padding=(1,1,1))
        self.conv4b = nn.Conv3d(512, 512, kernel_size=(3,3,3), padding=(1,1,1))
        self.pool4  = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2))
        self.conv5a = nn.Conv3d(512, 512, kernel_size=(3,3,3), padding=(1,1,1))
        self.conv5b = nn.Conv3d(512, 512, kernel_size=(3,3,3), padding=(1,1,1))
        self.pool5  = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2), padding=(0,1,1))
        self.relu    = nn.ReLU(inplace=True)
        self.fc6     = nn.Linear(8192, 4096)  # matches pretrained checkpoint
        self.fc7     = nn.Linear(4096, 64)    # penultimate = 64 per paper
        self.fc8     = nn.Linear(64, 1)       # regression output
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.relu(self.conv1(x));  x = self.pool1(x)
        x = self.relu(self.conv2(x));  x = self.pool2(x)
        x = self.relu(self.conv3a(x))
        x = self.relu(self.conv3b(x)); x = self.pool3(x)
        x = self.relu(self.conv4a(x))
        x = self.relu(self.conv4b(x)); x = self.pool4(x)
        x = self.relu(self.conv5a(x))
        x = self.relu(self.conv5b(x)); x = self.pool5(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(self.relu(self.fc6(x)))
        x = self.dropout(self.relu(self.fc7(x)))
        return self.fc8(x)

model = C3D().to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
with torch.no_grad():
    out = model(torch.zeros(2, 3, 16, 112, 112).to(device))
    print(f'Forward pass OK — output: {out.shape}')
    print(f'Flattened size after pool5: {model.fc6.in_features}')

Parameters: 61,476,737
Forward pass OK — output: torch.Size([2, 1])
Flattened size after pool5: 8192


In [16]:
# CELL 10: Load pretrained C3D weights (Sports-1M)
# conv layers:  features.N  -> conv1, conv2, conv3a etc
# fc6 and fc7:  classifier.0 and classifier.3 now match in shape (8192)
# fc8:          skipped — freshly initialised for regression

PRETRAINED_PATH = '/content/drive/MyDrive/c3d-pretrained.pth'

FEATURE_MAP = {
    'features.0.':  'conv1.',
    'features.3.':  'conv2.',
    'features.6.':  'conv3a.',
    'features.8.':  'conv3b.',
    'features.11.': 'conv4a.',
    'features.13.': 'conv4b.',
    'features.16.': 'conv5a.',
    'features.18.': 'conv5b.',
    'classifier.0.': 'fc6.',
    'classifier.3.': 'fc7.',
}

def load_pretrained_conv(model, path):
    if not os.path.exists(path):
        print('WARNING: pretrained weights not found — training from scratch.')
        return
    ckpt = torch.load(path, map_location=device)
    if isinstance(ckpt, dict) and 'state_dict' in ckpt:
        ckpt = ckpt['state_dict']
    model_dict = model.state_dict()
    matched = {}
    for k, v in ckpt.items():
        new_k = k
        for old_prefix, new_prefix in FEATURE_MAP.items():
            if k.startswith(old_prefix):
                new_k = k.replace(old_prefix, new_prefix)
                break
        # skip fc8 — wrong task (classification vs regression)
        if 'fc8' in new_k:
            continue
        if new_k in model_dict and model_dict[new_k].shape == v.shape:
            matched[new_k] = v
    model_dict.update(matched)
    model.load_state_dict(model_dict)
    print(f'Pretrained layers loaded: {len(matched)} tensors matched')
    for k in matched:
        print(f'  {k}')

load_pretrained_conv(model, PRETRAINED_PATH)

Pretrained layers loaded: 18 tensors matched
  conv1.weight
  conv1.bias
  conv2.weight
  conv2.bias
  conv3a.weight
  conv3a.bias
  conv3b.weight
  conv3b.bias
  conv4a.weight
  conv4a.bias
  conv4b.weight
  conv4b.bias
  conv5a.weight
  conv5a.bias
  conv5b.weight
  conv5b.bias
  fc6.weight
  fc6.bias


In [19]:
# CELL 11: Build datasets and DataLoader
train_dataset = AVEC2014Train(train_items)
val_dataset   = AVEC2014Eval(val_items)
test_dataset  = AVEC2014Eval(test_items)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)
print(f'Batches per epoch: {len(train_loader)}')

Training clips: 37961
Eval videos: 50
Eval videos: 50
Batches per epoch: 2373


In [20]:
# CELL 12: Training setup and evaluation
# Loss: nn.L1Loss() — supervisor specified, paper faithful

LR           = 1e-4
WEIGHT_DECAY = 1e-4
SAVE_PATH    = '/content/drive/MyDrive/c3d_avec2014_best.pth'
criterion    = nn.L1Loss()   # supervisor specified


def evaluate_videos(dataset, model):
    model.eval()
    y_true, y_pred, stems = [], [], []
    with torch.no_grad():
        for clips, label, stem in dataset:
            clips      = clips.to(device)
            preds      = model(clips).squeeze(1)
            video_pred = preds.mean().item() * LABEL_STD + LABEL_MEAN
            y_true.append(label.item())
            y_pred.append(video_pred)
            stems.append(stem)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mae    = np.mean(np.abs(y_true - y_pred))
    rmse   = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return mae, rmse, y_true, y_pred, stems


def train_stage(name, n_epochs, optimizer, patience, save_path):
    best_val_mae = float('inf')
    no_improve   = 0
    for epoch in range(1, n_epochs + 1):
        model.train()
        total_loss, n_batches = 0.0, 0
        for clips, targets in train_loader:
            clips   = clips.to(device)
            targets = targets.to(device).view(-1, 1)
            loss    = criterion(model(clips), targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches  += 1
        avg_loss = total_loss / n_batches
        val_mae, val_rmse, _, _, _ = evaluate_videos(val_dataset, model)
        print(f'[{name}] Epoch {epoch:3d} | Train L1: {avg_loss:.4f} | Val MAE: {val_mae:.4f} | Val RMSE: {val_rmse:.4f}')
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            no_improve   = 0
            torch.save(model.state_dict(), save_path)
            print(f'  -> Best saved (Val MAE={best_val_mae:.4f})')
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stopping at epoch {epoch}.')
                break
    return best_val_mae

In [ ]:
# CELL 13: Full fine-tuning
# All layers unfrozen from epoch 1
# lr=1e-5 chosen because fc6/fc7 are now loaded from pretrained weights
# (lower lr protects them from being overwritten too quickly)

load_pretrained_conv(model, PRETRAINED_PATH)

for p in model.parameters():
    p.requires_grad = True

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-5,
    weight_decay=WEIGHT_DECAY
)

print('=== Full fine-tuning ===')
train_stage('FullFT', n_epochs=60, optimizer=optimizer, patience=15, save_path=SAVE_PATH)

Pretrained layers loaded: 18 tensors matched
  conv1.weight
  conv1.bias
  conv2.weight
  conv2.bias
  conv3a.weight
  conv3a.bias
  conv3b.weight
  conv3b.bias
  conv4a.weight
  conv4a.bias
  conv4b.weight
  conv4b.bias
  conv5a.weight
  conv5a.bias
  conv5b.weight
  conv5b.bias
  fc6.weight
  fc6.bias
=== Full fine-tuning ===
...[FullFT] Epoch   1 | Train L1: 0.2693 | Val MAE: 8.0137 | Val RMSE: 9.9415
  -> Best saved (Val MAE=8.0137)
....[FullFT] Epoch   2 | Train L1: 0.1914 | Val MAE: 7.5921 | Val RMSE: 9.6902
  -> Best saved (Val MAE=7.5921)
....[FullFT] Epoch   3 | Train L1: 0.1796 | Val MAE: 7.5662 | Val RMSE: 9.5737
  -> Best saved (Val MAE=7.5662)
....[FullFT] Epoch   4 | Train L1: 0.1710 | Val MAE: 7.5387 | Val RMSE: 9.6408
  -> Best saved (Val MAE=7.5387)
...[FullFT] Epoch   5 | Train L1: 0.1625 | Val MAE: 7.5644 | Val RMSE: 9.5527
....[FullFT] Epoch   6 | Train L1: 0.1574 | Val MAE: 7.4915 | Val RMSE: 9.5543
  -> Best saved (Val MAE=7.4915)
....[FullFT] Epoch   7 | Train L1

np.float64(7.306709752439052)

In [ ]:
# RECOVERY CELL — only run if session disconnected mid-training
# Re-run Cells 1-12 first, then run this instead of Cell 13

# load_pretrained_conv(model, PRETRAINED_PATH)
# model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
# for p in model.parameters():
#     p.requires_grad = True
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-5, weight_decay=WEIGHT_DECAY)
# print('=== Resuming full fine-tuning ===')
# train_stage('FullFT', n_epochs=60, optimizer=optimizer, patience=15, save_path=SAVE_PATH)

In [25]:
# CELL 14 replacement — memory-efficient evaluation
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
model.eval()
print(f'Loaded: {SAVE_PATH}')

def evaluate_videos_chunked(dataset, model, chunk_size=32):
    y_true, y_pred, stems = [], [], []
    with torch.no_grad():
        for clips, label, stem in dataset:
            # clips: (N_clips, 3, 16, 112, 112)
            # process in chunks to avoid OOM
            all_preds = []
            for i in range(0, len(clips), chunk_size):
                chunk = clips[i:i+chunk_size].to(device)
                preds = model(chunk).squeeze(1)
                all_preds.append(preds.cpu())
                torch.cuda.empty_cache()
            video_pred = torch.cat(all_preds).mean().item() * LABEL_STD + LABEL_MEAN
            y_true.append(label.item())
            y_pred.append(video_pred)
            stems.append(stem)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mae    = np.mean(np.abs(y_true - y_pred))
    rmse   = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return mae, rmse, y_true, y_pred, stems

val_mae,  val_rmse,  vt, vp, vstems = evaluate_videos_chunked(val_dataset,  model)
test_mae, test_rmse, tt, tp, tstems = evaluate_videos_chunked(test_dataset, model)

print(f'\nVal  MAE={val_mae:.4f}  RMSE={val_rmse:.4f}   (paper: 5.53 / 7.71)')
print(f'Test MAE={test_mae:.4f}  RMSE={test_rmse:.4f}   (paper: 5.86 / 7.49)')
print(f'\nPrediction range on val:')
print(f'  Predicted mean: {np.mean(vp):.2f}  std: {np.std(vp):.2f}  min: {np.min(vp):.2f}  max: {np.max(vp):.2f}')
print(f'  True      mean: {np.mean(vt):.2f}  std: {np.std(vt):.2f}  min: {np.min(vt):.0f}  max: {np.max(vt):.0f}')

Loaded: /content/drive/MyDrive/c3d_avec2014_best.pth

Val  MAE=7.3067  RMSE=9.3240   (paper: 5.53 / 7.71)
Test MAE=7.9345  RMSE=9.8987   (paper: 5.86 / 7.49)

Prediction range on val:
  Predicted mean: 14.24  std: 7.38  min: 1.97  max: 29.46
  True      mean: 14.50  std: 11.48  min: 0  max: 43


In [26]:
# CELL 15: Gender breakdown
def gender_metrics(y_true, y_pred, stems, gender_map, split_name):
    if not gender_map:
        print('No gender map — skipping.')
        return
    groups = {'F': ([], []), 'M': ([], [])}
    for yt, yp, s in zip(y_true, y_pred, stems):
        g = gender_map.get(s)
        if g in groups:
            groups[g][0].append(yt)
            groups[g][1].append(yp)
    print(f'\n--- {split_name} gender breakdown ---')
    for g, (yt_g, yp_g) in groups.items():
        if not yt_g:
            continue
        yt_g = np.array(yt_g)
        yp_g = np.array(yp_g)
        lbl  = 'Female' if g == 'F' else 'Male'
        print(f'  {lbl} (N={len(yt_g)})  MAE={np.mean(np.abs(yt_g-yp_g)):.4f}  RMSE={np.sqrt(np.mean((yt_g-yp_g)**2)):.4f}')

gender_metrics(vt, vp, vstems, gender_map, 'Val')
gender_metrics(tt, tp, tstems, gender_map, 'Test')


--- Val gender breakdown ---
  Female (N=20)  MAE=7.4010  RMSE=9.3382
  Male (N=30)  MAE=7.2438  RMSE=9.3145

--- Test gender breakdown ---
  Female (N=15)  MAE=7.6472  RMSE=9.8613
  Male (N=35)  MAE=8.0577  RMSE=9.9147


In [27]:
# CELL 16: Save predictions CSV for FUQ calibration
rows = [
    {'stem': s, 'y_true': yt, 'y_pred': yp, 'gender': gender_map.get(s, '?')}
    for yt, yp, s in zip(tt, tp, tstems)
]
out_csv = '/content/drive/MyDrive/avec2014_test_predictions.csv'
pd.DataFrame(rows).to_csv(out_csv, index=False)
print(f'Saved: {out_csv}')

Saved: /content/drive/MyDrive/avec2014_test_predictions.csv
